# Homework #4: Model Building and Tracking with MLflow

**Course:** GR5069 - Data Pipeline in Practice
**Student:** Bruce (UNI: jf3774)

## Task
Build an ML model on the F1 dataset, track runs with MLflow, run at least 10 experiments
with different hyperparameters, and select the best model.

## Modeling Problem
**Binary classification:** predict whether a driver will finish on the **podium**
(position 1, 2, or 3) in a race, given pre-race and race-context features such as
starting grid position, constructor, driver, and circuit.

**Model:** `RandomForestClassifier` (scikit-learn) with tunable hyperparameters.
**Tracking:** MLflow (hyperparameters, metrics, model, plots + CSV artifacts).

In [0]:
%pip install mlflow -q
dbutils.library.restartPython()

## 1. Imports and MLflow setup

In [0]:
import os
import itertools
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import functions as F

import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

# Set (or create) an MLflow experiment so runs are grouped on the MLflow homepage.
EXPERIMENT_NAME = "/Users/jf3774@columbia.edu/gr5069_hw4_f1_podium"
try:
    mlflow.set_experiment(EXPERIMENT_NAME)
except Exception:
    # Fallback: use a workspace-relative path if the above isn't valid in this workspace.
    mlflow.set_experiment("gr5069_hw4_f1_podium")

## 2. Load F1 data from Unity Catalog Volumes

All CSVs live at `/Volumes/gr5069/raw/f1_data/`, read with `header=True` and all
columns as strings (per course convention). We cast numeric columns after loading.

In [0]:
BASE_PATH = "/Volumes/gr5069/raw/f1_data"

results     = spark.read.csv(f"{BASE_PATH}/results.csv",     header=True)
races       = spark.read.csv(f"{BASE_PATH}/races.csv",       header=True)
drivers     = spark.read.csv(f"{BASE_PATH}/drivers.csv",     header=True)
constructors = spark.read.csv(f"{BASE_PATH}/constructors.csv", header=True)
circuits    = spark.read.csv(f"{BASE_PATH}/circuits.csv",    header=True)

print("results    :", results.count(),    "rows")
print("races      :", races.count(),      "rows")
print("drivers    :", drivers.count(),    "rows")
print("constructors:", constructors.count(), "rows")
print("circuits   :", circuits.count(),   "rows")

## 3. Build modeling dataframe

Join results with races, drivers, constructors, and circuits. Cast numeric columns,
engineer the binary target `podium`, and keep a compact feature set.

In [0]:
# Disambiguate columns BEFORE joining: drivers and constructors both have "nationality",
# and races/circuits both have "url". Rename them on the right-side dataframes.
drivers_r = (drivers
             .withColumnRenamed("nationality", "driver_nationality")
             .withColumnRenamed("url", "driver_url"))
constructors_r = (constructors
                  .withColumnRenamed("nationality", "constructor_nationality")
                  .withColumnRenamed("url", "constructor_url")
                  .withColumnRenamed("name", "constructor_name"))
races_r = races.withColumnRenamed("url", "race_url").withColumnRenamed("name", "race_name")
circuits_r = (circuits
              .withColumnRenamed("url", "circuit_url")
              .withColumnRenamed("name", "circuit_name"))

df = (
    results
    .join(races_r,        on="raceId",        how="inner")
    .join(drivers_r,      on="driverId",      how="inner")
    .join(constructors_r, on="constructorId", how="inner")
    .join(circuits_r,     on="circuitId",     how="inner")
)

# Cast the columns we need to numeric types (all are strings on load).
df = (
    df
    .withColumn("grid",          F.col("grid").cast("int"))
    .withColumn("positionOrder", F.col("positionOrder").cast("int"))
    .withColumn("laps",           F.col("laps").cast("int"))
    .withColumn("year",           F.col("year").cast("int"))
    .withColumn("round",          F.col("round").cast("int"))
    .withColumn("lat",            F.col("lat").cast("double"))
    .withColumn("lng",            F.col("lng").cast("double"))
    .withColumn("alt",            F.col("alt").cast("double"))
)

# Target: podium finish (positionOrder in {1,2,3}).
df = df.withColumn(
    "podium",
    F.when(F.col("positionOrder").isin(1, 2, 3), F.lit(1)).otherwise(F.lit(0)),
)

# Keep a focused feature set (using the disambiguated names).
feature_cols = [
    "grid", "laps", "year", "round",
    "lat", "lng", "alt",
    "driver_nationality",   # from drivers
    "constructorRef",       # constructor identifier
    "circuitRef",           # circuit identifier
]
label_col = "podium"

# Drop pit-lane starts (grid==0) — anomalous, not a real starting position.
df = df.filter(F.col("grid") > 0)

model_sdf = df.select(*feature_cols, label_col).dropna()
print("Modeling rows:", model_sdf.count())
model_sdf.groupBy(label_col).count().show()


## 4. Convert to pandas and encode categoricals

sklearn needs a numeric matrix. We one-hot encode the three categorical columns.
The dataset is small enough (~26k rows) to fit comfortably in pandas.

In [0]:
pdf = model_sdf.toPandas()
print("Pandas shape:", pdf.shape)

categorical_cols = ["driver_nationality", "constructorRef", "circuitRef"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

X = pd.get_dummies(pdf[feature_cols], columns=categorical_cols, drop_first=True)
y = pdf[label_col].astype(int).values

print("Feature matrix shape after one-hot:", X.shape)
print("Podium rate:", y.mean().round(4))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 5. Training + logging function

For every run we log:
- **Hyperparameters** (all tunable RF params + dataset sizes)
- **The model** itself (`mlflow.sklearn.log_model`)
- **Every reasonable metric** for binary classification
- **Artifacts**: a confusion-matrix plot, ROC curve plot, and a per-run predictions CSV
(three artifacts; the rubric requires at least two).

In [0]:
def train_and_log(params: dict, run_name: str) -> dict:
    """Train a RandomForestClassifier and log everything to MLflow.

    Returns a dict summarizing the run for later comparison.
    """
    with mlflow.start_run(run_name=run_name) as run:
        # ---- Params ----
        mlflow.log_params(params)
        mlflow.log_param("n_train", X_train.shape[0])
        mlflow.log_param("n_test", X_test.shape[0])
        mlflow.log_param("n_features", X_train.shape[1])

        # ---- Fit ----
        model = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
        model.fit(X_train, y_train)

        # ---- Predict ----
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        # ---- Metrics (every reasonable one for binary classification) ----
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        metrics = {
            "accuracy":            accuracy_score(y_test, y_pred),
            "balanced_accuracy":   balanced_accuracy_score(y_test, y_pred),
            "precision":           precision_score(y_test, y_pred, zero_division=0),
            "recall":              recall_score(y_test, y_pred, zero_division=0),
            "f1":                  f1_score(y_test, y_pred, zero_division=0),
            "roc_auc":             roc_auc_score(y_test, y_proba),
            "average_precision":   average_precision_score(y_test, y_proba),
            "log_loss":            log_loss(y_test, y_proba, labels=[0, 1]),
            "mcc":                 matthews_corrcoef(y_test, y_pred),
            "cohen_kappa":         cohen_kappa_score(y_test, y_pred),
            "true_positives":      int(tp),
            "true_negatives":      int(tn),
            "false_positives":     int(fp),
            "false_negatives":     int(fn),
            "specificity":         tn / (tn + fp) if (tn + fp) else 0.0,
            "train_accuracy":      model.score(X_train, y_train),
            "oob_score":           float(getattr(model, "oob_score_", np.nan))
                                   if params.get("oob_score") else float("nan"),
        }
        mlflow.log_metrics({k: v for k, v in metrics.items() if not np.isnan(v)})

        # ---- Artifacts ----
        with tempfile.TemporaryDirectory() as tmp:
            # Artifact 1: confusion-matrix heatmap
            cm_path = os.path.join(tmp, "confusion_matrix.png")
            fig, ax = plt.subplots(figsize=(5, 4))
            sns.heatmap(
                confusion_matrix(y_test, y_pred),
                annot=True, fmt="d", cmap="Blues",
                xticklabels=["no podium", "podium"],
                yticklabels=["no podium", "podium"],
                ax=ax,
            )
            ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
            ax.set_title(f"Confusion matrix — {run_name}")
            fig.tight_layout(); fig.savefig(cm_path, dpi=120); plt.close(fig)
            mlflow.log_artifact(cm_path)

            # Artifact 2: ROC curve
            roc_path = os.path.join(tmp, "roc_curve.png")
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            fig, ax = plt.subplots(figsize=(5, 4))
            ax.plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}")
            ax.plot([0, 1], [0, 1], "--", color="grey")
            ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
            ax.set_title(f"ROC — {run_name}"); ax.legend()
            fig.tight_layout(); fig.savefig(roc_path, dpi=120); plt.close(fig)
            mlflow.log_artifact(roc_path)

            # Artifact 3: predictions CSV
            csv_path = os.path.join(tmp, "test_predictions.csv")
            pd.DataFrame({
                "y_true": y_test,
                "y_pred": y_pred,
                "y_proba_podium": y_proba,
            }).to_csv(csv_path, index=False)
            mlflow.log_artifact(csv_path)

            # Artifact 4 (bonus): top-20 feature importances CSV
            fi_path = os.path.join(tmp, "feature_importances.csv")
            (pd.DataFrame({
                "feature": X_train.columns,
                "importance": model.feature_importances_,
            })
             .sort_values("importance", ascending=False)
             .head(20)
             .to_csv(fi_path, index=False))
            mlflow.log_artifact(fi_path)

        # ---- Model ----
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"[{run_name}] F1={metrics['f1']:.3f}  AUC={metrics['roc_auc']:.3f}  "
              f"acc={metrics['accuracy']:.3f}")

        return {"run_id": run.info.run_id, "run_name": run_name,
                **params, **metrics}

## 6. Run 10+ experiments with different hyperparameters

We vary `n_estimators`, `max_depth`, `min_samples_split`, `max_features`, and
`class_weight`. The list below gives 12 distinct configurations.

In [0]:
experiments = [
    ("Run 01 - baseline small",        {"n_estimators":  50, "max_depth":  5,   "min_samples_split": 2,  "max_features": "sqrt", "class_weight": None}),
    ("Run 02 - more trees shallow",    {"n_estimators": 100, "max_depth":  5,   "min_samples_split": 2,  "max_features": "sqrt", "class_weight": None}),
    ("Run 03 - medium depth",          {"n_estimators": 100, "max_depth": 10,   "min_samples_split": 2,  "max_features": "sqrt", "class_weight": None}),
    ("Run 04 - deeper trees",          {"n_estimators": 100, "max_depth": 15,   "min_samples_split": 2,  "max_features": "sqrt", "class_weight": None}),
    ("Run 05 - 200 trees depth 10",    {"n_estimators": 200, "max_depth": 10,   "min_samples_split": 2,  "max_features": "sqrt", "class_weight": None}),
    ("Run 06 - 200 trees depth 20",    {"n_estimators": 200, "max_depth": 20,   "min_samples_split": 5,  "max_features": "sqrt", "class_weight": None}),
    ("Run 07 - unlimited depth",       {"n_estimators": 300, "max_depth": None, "min_samples_split": 2,  "max_features": "sqrt", "class_weight": None}),
    ("Run 08 - log2 features",         {"n_estimators": 300, "max_depth": 20,   "min_samples_split": 10, "max_features": "log2", "class_weight": None}),
    ("Run 09 - balanced classes",      {"n_estimators": 200, "max_depth": 15,   "min_samples_split": 2,  "max_features": "sqrt", "class_weight": "balanced"}),
    ("Run 10 - balanced deep",         {"n_estimators": 300, "max_depth": 20,   "min_samples_split": 5,  "max_features": "sqrt", "class_weight": "balanced"}),
    ("Run 11 - balanced log2 large",   {"n_estimators": 500, "max_depth": 25,   "min_samples_split": 5,  "max_features": "log2", "class_weight": "balanced"}),
    ("Run 12 - tuned combo",           {"n_estimators": 400, "max_depth": None, "min_samples_split": 10, "max_features": "sqrt", "class_weight": "balanced"}),
]

results_records = []
for run_name, params in experiments:
    results_records.append(train_and_log(params, run_name))

results_df = pd.DataFrame(results_records)
display(results_df.sort_values("f1", ascending=False))


## 7. Compare all runs visually

A side-by-side view of **F1, ROC AUC, and Accuracy** across all 12 runs makes it
easy to see which hyperparameter choices paid off. The best run is highlighted in
orange in every panel, so the selection is visually justified across all three
metrics at once.


In [0]:
# Visual comparison of F1, ROC AUC, and Accuracy across all 12 runs
sorted_df = results_df.sort_values("f1", ascending=True).reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

metrics_to_plot = [
    ("f1",       "F1 score",  "steelblue"),
    ("roc_auc",  "ROC AUC",   "seagreen"),
    ("accuracy", "Accuracy",  "indianred"),
]

best_run_name = results_df.sort_values(["f1", "roc_auc"], ascending=False).iloc[0]["run_name"]

for ax, (metric, title, color) in zip(axes, metrics_to_plot):
    bars = ax.barh(sorted_df["run_name"], sorted_df[metric], color=color)
    # Highlight the best run in orange
    for bar, name in zip(bars, sorted_df["run_name"]):
        if name == best_run_name:
            bar.set_color("darkorange")
    ax.set_xlabel(title)
    ax.set_title(title)
    ax.set_xlim(0, 1.0)
    for i, val in enumerate(sorted_df[metric]):
        ax.text(val + 0.01, i, f"{val:.3f}", va="center", fontsize=8)

fig.suptitle("Model comparison across all 12 runs (best run in orange)", fontsize=13)
fig.tight_layout()
plt.show()


## 8. Select the best run

We rank by **F1 score** on the held-out test set because the podium class is
imbalanced (~15% positives), so F1 balances precision and recall better than
raw accuracy. We also report ROC AUC as a secondary check.

In [0]:
best = results_df.sort_values(["f1", "roc_auc"], ascending=False).iloc[0]
print("Best run:")
print(best[["run_name", "run_id", "n_estimators", "max_depth",
            "min_samples_split", "max_features", "class_weight",
            "f1", "roc_auc", "accuracy", "precision", "recall"]])

### Why Run 12 is the best model

**Result:** `Run 12 - tuned combo` wins on every metric that matters here.

| Metric | Run 12 | Best of the rest |
|---|---|---|
| F1 | **0.673** | 0.651 (Run 11) |
| ROC AUC | **0.938** | 0.934 (Run 10) |
| Accuracy | **0.901** | 0.901 (Run 07, tied) |
| Precision | 0.608 | — |
| Recall | 0.753 | — |

**Hyperparameters:** `n_estimators=400`, `max_depth=None`, `min_samples_split=10`,
`max_features="sqrt"`, `class_weight="balanced"`.

**Why F1 is the right selection criterion.** Only about 13.5% of rows are podium finishes (3,397 / 25,121), so a model that predicts "no podium" for everything gets about 86.5% accuracy while being completely useless. This is exactly what happened in Runs 01, 02, 05, and 08 — high accuracy (about 0.87), but F1 ≈ 0 because the shallow / class-unaware trees never predict the minority class. F1 jointly penalizes missed podiums (recall) and false podium calls (precision), so it's the honest scoreboard.

**What the run-by-run progression shows:**
1. **Depth matters a lot without class weighting.** Going from `max_depth=5` (Runs 01–02,
   F1=0.00) to `max_depth=20` (Run 06, F1=0.42) to `max_depth=None` (Run 07, F1=0.57) steadily
   improves the model's willingness to predict the minority class.
2. **`class_weight="balanced"` is the single biggest lever.** Adding it jumps F1 from 0.57
   (Run 07) to 0.63–0.65 (Runs 09–11) — the loss function now actually cares about podium
   finishes instead of being dominated by the 86.5% no-podium class.
3. **Run 12 combines the winning ingredients:** unlimited depth (from Run 07), balanced
   class weight (from Runs 09–11), and a larger forest (400 trees) with `min_samples_split=10`
   for modest regularization. That combination gives the best F1 (0.673) *and* the best
   ROC AUC (0.938) — so the choice is unambiguous regardless of which metric you privilege.

**Operational read-out.** At the default 0.5 threshold, Run 12 correctly identifies ~75% of
podium finishes (recall = 0.753) with ~61% precision — meaning when it flags a driver as a
likely podium, it's right 6 times out of 10. For a sports-prediction task with an ~86.5/13.5
class split, that's a strong result.
